# DataPilot — демонстрація на реальних даних

Лабораторна робота №4. Соломка Андрій, ІМ-31.

Ноутбук показує роботу інструменту на зрізі навчальної бази мобільного застосунку
за 2024-04-05 – 2024-05-11.

Клітинки нижче виконані **без API-ключа** (`use_llm=False`), тому видно
детерміновану частину — усе, що рахує pandas. Саме ці числа потім передаються
моделі; вона не рахує нічого сама. Щоб побачити текстові інтерпретації,
достатньо задати `ANTHROPIC_API_KEY` і прибрати `use_llm=False`.

In [1]:
import sys, json, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

users  = pd.read_csv("../data/users.csv")
daily  = pd.read_csv("../data/daily_metrics.csv")
pays   = pd.read_csv("../data/payments.csv")

print("users        ", users.shape)
print("daily_metrics", daily.shape)
print("payments     ", pays.shape)
users.head(3)

users         (60000, 18)
daily_metrics (7825, 11)
payments      (11330, 7)


,user_id,reg_date,app,country_code,country_name,gender,age,group_id,sessions_cnt,total_duration_min,active_days,activities_cnt,contacts_cnt,payments_cnt,revenue_usd,is_payer,age_group,avg_session_min
0,20001,2024-04-05,ios,EC,Ecuador,m,27.0,not_in_test,2.0,138.4,1.0,0.0,0.0,0.0,0.0,0,25-34,69.20
1,20014,2024-04-05,ios,US,United States of America,f,30.0,not_in_test,1.0,33.9,1.0,0.0,0.0,0.0,0.0,0,25-34,33.90
2,20016,2024-04-05,ios,BR,Brazil,m,52.0,not_in_test,4.0,142.2,1.0,0.0,0.0,0.0,0.0,0,45-54,35.55


## Функція 1. Автоматичний EDA

`auto_eda` повертає два блоки: `facts` — усе, що порахував pandas, і `llm` —
інтерпретацію моделі. Нижче — лише `facts`.

In [2]:
from datapilot.eda import auto_eda

res = auto_eda(daily, name="daily_metrics.csv", use_llm=False)
f = res["facts"]
print(f"Рядків: {f['rows']:,}  Колонок: {f['columns']}  Дублікатів: {f['duplicated_rows']}")

pd.DataFrame([{ "колонка": c["column"], "тип": c["kind"],
                "пропуски %": c["missing_pct"], "унікальних": c["unique"],
                "викидів IQR": c.get("outliers_iqr") }
              for c in f["columns_profile"]])

Рядків: 7,825  Колонок: 11  Дублікатів: 0


,колонка,тип,пропуски %,унікальних,викидів IQR
0,date,datetime,0.0,37,NaN
1,app,categorical,0.0,5,NaN
2,country_name,categorical,0.0,196,NaN
3,sessions,numeric,0.0,777,1282.0
4,session_minutes,numeric,0.0,4956,1263.0
5,active_users,numeric,0.0,542,1355.0
6,new_users,numeric,0.0,323,1238.0
7,revenue_usd,numeric,0.0,671,1411.0
8,payers,numeric,0.0,67,1411.0
9,activities,numeric,0.0,2670,1369.0


In [3]:
for flag in f["quality_flags"]:
    print(" •", flag)

 • Колонка 'sessions': 1282 викидів за IQR (16.38%)
 • Колонка 'session_minutes': 1263 викидів за IQR (16.14%)
 • Колонка 'active_users': 1355 викидів за IQR (17.32%)
 • Колонка 'new_users': 1238 викидів за IQR (15.82%)
 • Колонка 'revenue_usd': 81.97% нулів — перевірте, чи це не пропуски
 • Колонка 'revenue_usd': 1411 викидів за IQR (18.03%)
 • Колонка 'payers': 81.97% нулів — перевірте, чи це не пропуски
 • Колонка 'payers': 1411 викидів за IQR (18.03%)
 • Колонка 'activities': 1369 викидів за IQR (17.5%)
 • Колонка 'avg_session_min': 1530 викидів за IQR (19.55%)


Прапорці якості рахує код, а не модель. Наприклад, `81.97% нулів` у
`revenue_usd` — це не пропуски, а очікувана картина: більшість пар
«день × країна» не має платежів. Саме такі речі модель і має інтерпретувати,
а не вигадувати.

## Функція 2. Матеріал для гіпотез

Перед тим як просити гіпотези, код рахує зрізи: середні метрики по кожному
сегменту, розкид між найкращою і найгіршою групою, кореляції та денні аномалії.

In [4]:
from datapilot.insights import build_fact_bundle

bundle = build_fact_bundle(users, name="users.csv")
app_cut = next(c for c in bundle["segment_cuts"] if c["segment"] == "app")
pd.DataFrame(app_cut["groups"])[["group", "rows", "share_pct",
                                 "avg_revenue_usd", "avg_is_payer",
                                 "avg_sessions_cnt", "avg_total_duration_min"]]

,group,rows,share_pct,avg_revenue_usd,avg_is_payer,avg_sessions_cnt,avg_total_duration_min
0,android,387,0.65,1.8413,0.1137,15.9974,512.5811
1,desktop,1051,1.75,0.6880,0.0618,15.2093,502.1699
2,ios,2610,4.35,0.5758,0.0475,5.0490,162.5787
3,mobile,55951,93.25,0.4182,0.0355,7.3103,217.8648
4,unknown,1,0.00,0.0000,0.0000,96.0000,2096.7000


In [5]:
# розкид між найкращою і найгіршою групою — звідси беруться найцікавіші гіпотези
json.loads(json.dumps(app_cut["spread"], default=str))

{'revenue_usd': {'best_group': 'android',
  'best_value': 1.8413,
  'worst_group': 'unknown',
  'worst_value': 0.0,
  'ratio': None},
 'is_payer': {'best_group': 'android',
  'best_value': 0.1137,
  'worst_group': 'unknown',
  'worst_value': 0.0,
  'ratio': None},
 'payments_cnt': {'best_group': 'android',
  'best_value': 0.1628,
  'worst_group': 'unknown',
  'worst_value': 0.0,
  'ratio': None},
 'total_duration_min': {'best_group': 'unknown',
  'best_value': 2096.7,
  'worst_group': 'ios',
  'worst_value': 162.5787,
  'ratio': 12.9},
 'sessions_cnt': {'best_group': 'unknown',
  'best_value': 96.0,
  'worst_group': 'ios',
  'worst_value': 5.049,
  'ratio': 19.01},
 'avg_session_min': {'best_group': 'desktop',
  'best_value': 31.3892,
  'worst_group': 'unknown',
  'worst_value': 21.84,
  'ratio': 1.44}}

## Функція 3. Питання природною мовою → pandas

Модель повертає код, код показується користувачеві й виконується в пісочниці.
Нижче — приклад коду саме в тому вигляді, в якому його повертає модель, і його
виконання через `run_code`.

In [6]:
from datapilot.nl2pandas import run_code, schema_of

generated = """
result = (df.groupby('country_name')['amount_usd'].sum()
            .sort_values(ascending=False).head(10).round(2).to_frame('revenue_usd'))
"""
run_code(generated, pays)

,revenue_usd
country_name,
Ukraine,22954.52
United States of America,19141.17
South Africa,6053.40
Chile,5513.70
Spain,4895.21
United Kingdom of Great Britain and Northern Ireland,3966.89
Kazakhstan,3669.68
Argentina,2957.92
Germany,2689.54


In [7]:
# пісочниця відхиляє небезпечний код
from datapilot.nl2pandas import UnsafeCode
for bad in ["import os\nresult = os.listdir('/')", "result = open('/etc/passwd').read()"]:
    try:
        run_code(bad, pays)
    except UnsafeCode as exc:
        print("відхилено:", exc)

відхилено: У згенерованому коді є заборонена конструкція: 'import'
відхилено: У згенерованому коді є заборонена конструкція: 'open('


## Функція 4. Пояснення зміни метрики

Порівнюємо два періоди: до 19 квітня і після. Код рахує внесок кожного
сегмента у зміну, модель лише формулює гіпотезу причини.

In [8]:
from datapilot.anomaly import decompose

facts = decompose(daily, metric="new_users", date_col="date",
                  base=("2024-04-05", "2024-04-19"),
                  compare=("2024-04-20", "2024-05-11"),
                  dims=["app"])
print(f"Було:  {facts['base_period']['total']:,.0f}")
print(f"Стало: {facts['compare_period']['total']:,.0f}")
print(f"Зміна: {facts['absolute_change']:,.0f} ({facts['relative_change_pct']}%)")
pd.DataFrame(facts["by_dimension"]["app"])

Було:  27,204
Стало: 112,589
Зміна: 85,385 (313.87%)


,group,base,compare,change,change_pct,contribution_pct
0,mobile,17096.0,111109.0,94013.0,549.91,110.10
1,ios,8710.0,89.0,-8621.0,-98.98,-10.10
2,android,775.0,79.0,-696.0,-89.81,-0.82
3,desktop,623.0,1312.0,689.0,110.59,0.81
4,unknown,0.0,0.0,0.0,NaN,0.00


Загальний притік виріс у чотири рази — і на цьому тлі легко не помітити, що
реєстрації на **iOS впали на 98.98%**. Декомпозиція по внесках показує це одразу:
iOS дає −10.1% до загальної зміни, тоді як mobile — +110.1%.

In [9]:
facts_rev = decompose(daily, metric="revenue_usd", date_col="date",
                      base=("2024-04-05", "2024-04-19"),
                      compare=("2024-04-20", "2024-05-11"),
                      dims=["country_name"])
pd.DataFrame(facts_rev["by_dimension"]["country_name"])

,group,base,compare,change,change_pct,contribution_pct
0,Ukraine,3701.50,19253.02,15551.52,420.14,23.03
1,United States of America,4454.62,14686.55,10231.93,229.69,15.15
2,South Africa,472.06,5581.34,5109.28,1082.34,7.57
3,Chile,629.99,4883.71,4253.72,675.20,6.30
4,Spain,335.61,4559.60,4223.99,1258.60,6.26
5,United Kingdom of Great Britain and Northern I...,319.06,3647.83,3328.77,1043.31,4.93


## Функція 5. Метрики для executive summary

Усе, що піде в текст для керівника, рахується заздалегідь і зберігається у JSON.

In [10]:
metrics = json.load(open("metrics.json"))
metrics

{'period': {'from': '2024-04-05', 'to': '2024-05-11', 'days': 37},
 'new_users_total': 139793,
 'revenue_usd_total': 93749.58,
 'payers_total': 2218,
 'payer_rate_pct': 3.7,
 'arppu_usd': 11.87,
 'avg_session_min': 30.09,
 'dau_first_day': 794,
 'dau_peak': 33071,
 'dau_last_day': 6367,
 'new_users_by_app': {'android': 854,
  'desktop': 1935,
  'ios': 8799,
  'mobile': 128205,
  'unknown': 0},
 'ios_new_users_before_apr20': 8710,
 'ios_new_users_after_apr20': 89,
 'revenue_top5_countries': {'Ukraine': 22954.52,
  'United States of America': 19141.17,
  'South Africa': 6053.4,
  'Chile': 5513.7,
  'Spain': 4895.21}}

## Захист від галюцинацій

Верифікатор витягує з тексту моделі всі числа і звіряє їх із фактами.
Округлення й перерахунок у тисячі чи відсотки допускаються, вигадані числа — ні.

In [11]:
from datapilot.verify import verify_text

ok   = "За період зареєструвалось 139 793 користувачі, виручка 93 749.58 $."
bad  = "За період зареєструвалось 139 793 користувачі, конверсія в платіж 4.1%."

for text in (ok, bad):
    v = verify_text(text, metrics)
    status = "OK" if v["ok"] else f"НЕПІДТВЕРДЖЕНО {v['unverified']}"
    print(f"{status:<28} ← {text}")

OK                           ← За період зареєструвалось 139 793 користувачі, виручка 93 749.58 $.
НЕПІДТВЕРДЖЕНО [4.1]         ← За період зареєструвалось 139 793 користувачі, конверсія в платіж 4.1%.


Реальна конверсія у платіж — `payer_rate_pct = 3.7`. Число `4.1` правдоподібне
і стоїть поруч із правильним, тому без автоматичної перевірки воно спокійно
пішло б далі у звіт. Саме такі помилки найнебезпечніші.

## Підсумок

Інструмент покриває п'ять функцій із завдання плюс власну — верифікацію чисел.
Поділ відповідальності всюди однаковий: **pandas рахує, LLM пояснює, код
перевіряє**. Модель жодного разу не отримує сирий CSV і не виконує арифметику,
тому помилитися в числі вона може тільки в тексті — і там її ловить верифікатор.